In [0]:
/Volumes/retaildataplatform/bronze/raw_data/sqlserver_customers/load_ date=2026-09-20/%pip install pymongo

In [0]:
dbutils.library.restartPython()



In [0]:
import json
from datetime import date
from common_utils.logging import get_logger
from common_utils.ingestor import read_jdbc,write_raw

#0.logger
logger = get_logger("sql-server-ingestion")


# 1  config path
dbutils.widgets.text("path","")
config_path= dbutils.widgets.get("path")

# 2 reading config 

with open(config_path,"r") as f:
    config = json.load(f)

# 3 Configuration

source_config= config["source"]
target_config=config["target"]
write_options=config["write_options"]

# 4 run date

run_date = date.today().isoformat()


logger.info("reading data from %s",source_config["dbtable"])

user=dbutils.secrets.get(scope="retail-platform-dev",key=source_config["user"])
password=dbutils.secrets.get(scope="retail-platform-dev",key=source_config["password"])


#5 read  data from sql server

df= read_jdbc(spark,source_config["url"],source_config["dbtable"],user,password)
logger.info("read  %s rows",df.count())
logger.info("sample data...........")
df.show()


#6 write from sql server

target_path=f"{target_config["base_path"]}/{target_config["folder"]}/load_date={run_date}"

write_raw(df,target_path,target_config["file_format"],target_config["mode"],write_options)

logger.info("data landed at %s",target_path)

In [0]:
from datetime import date

url = "jdbc:sqlserver://rivadata.database.windows.net:1433;databaseName=batch2;encrypt=true;trustServerCertificate=true;loginTimeout=90" 
user = "rivadata" 
password = "batch2@de"

dbtable = "retail.customers"

df = spark.read.format("jdbc") \
    .option("url",url  ) \
    .option("dbtable", dbtable) \
    .option("user", user) \
    .option("password", password) \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .load()


run_date = date.today().isoformat()
target_path=f"/Volumes/retaildataplatform/bronze/raw_data/sqlserver_customers/load_ date={run_date}"



df.write.mode('overwrite')\
        .option("header", "true")\
        .option("quoteAll","true")\
        .option("escape", '"')\
        .csv(target_path)   
        
df.display()


